In [21]:
curpage = 0
import re
from greek_normalisation.utils import (
     nfd, nfc, nfkc,
     strip_accents, count_accents, strip_last_accent, grave_to_acute,
     strip_last_accent_if_two, breathing_check, convert_to_2019
 )
pagebuf = {}
nauckpages = {}
wordindex = {}
xmlindex = {}
wordtots = {}
g2r = {
    'Α' : 'A',
    'Β' : 'B',
    'Ε' : 'E',
    'Ζ' : 'Z',
    'Η' : 'H',
    'Ι' : 'I',
    'Κ' : 'K',
    'Μ' : 'M',
    'Ν' : 'N',
    'Ο' : 'O',
    'Ρ' : 'P',
    'Τ' : 'T',
    'Χ' : 'X',
}
g2rreg = '(['
for foo in g2r:
    g2rreg = g2rreg + foo 
g2rreg = g2rreg + '])'

r2g = {}
for foo in g2r:
    r2g[g2r[foo]] = foo

    
for i in range(3,959):
    #if(i> 913 and i< 948):
        
    #    continue
    if(i<10):
        fname = '../../Downloads/nauck/000' + str(i) + '.txt'
    elif(i<100):
        fname = '../../Downloads/nauck/00' + str(i) + '.txt'
    elif(i<1000):
        fname = '../../Downloads/nauck/0' + str(i) + '.txt'
    else:
        fname = '../../Downloads/nauck/' + str(i) + '.txt'
    
    f = open(fname)
    curbuf = ''
    for l in f:
       # if(re.search('ΑΛΕΑΔΑΙ',l)):
       #     print('0',l)
        while(re.search('([Α-Ω])([A-Z])',l)):
            m = re.search('([Α-Ω])([A-Z])',l)
            if(m):
                #print(m[1],m[2],l)
                curkey = m[1] + m[2]
                curroman = m[1]
                if curroman in r2g:
                    cursub =  m[1] + r2g[curroman]
                else:
                    cursub = m[1] + '<1>' +  m[2]
                l = re.sub(curkey,cursub,l,1)
                #print(fname,'G+R',m[1],m[2],l,'\n')
        l = re.sub('<1>','',l)
        while(re.search('([0-9]+\s+|chol\.\s+)'+g2rreg+'\\b',l)):
            m = re.search('([0-9]+\s+|chol\.\s+)'+g2rreg+'\\b',l)
            if(m):
                #print(fname,'a',m[1],m[2],l)
                curkey = m[1] + m[2]
                cursub = m[1] + g2r[m[2]]
                l = re.sub(curkey,cursub,l,1)
       # if(re.search('ΑΛΕΑΔAI',l)):
       #     print('1a',l)
       # elif(re.search('ΑΛΕΑΔΑΙ',l)):
       #     print('stillok')
        while(re.search('\\b'+g2rreg+'(\\b|[:,\.a-zA-Z])',l)):
            m = re.search('\\b'+g2rreg+'(\\b|[:,\.a-zA-Z])',l)
            if(m):
                #print(m[1],m[2])
                curkey = m[1] + m[2]
                cursub = g2r[m[1]] + m[2] 
                l = re.sub(curkey,cursub,l,1)
       # if(re.search('ΑΛΕΑΔAI',l)):
       #     print('1',l)
        l = re.sub('\s+$','',l)
        if(re.search('(\\b[ΧΙ]+\\b)',l)):
            #print('rn',fname,l)
            while(re.search(g2rreg,l)):
                m = re.search(g2rreg,l)
                subval = g2r[m[1]]
                l = re.sub(m[1],subval,l)
            
        l = re.sub('\\bef\\b','cf',l)
        l = re.sub('[MΜ][AΑ]','MA',l)
        l = re.sub('˙','·',l)
        l = re.sub('·','·',l)
        l = re.sub('—','',l)
        l = re.sub('”,','’,',l)
        l = re.sub('^\s+','',l)
        l = re.sub('\s+','\n',l)
        l = re.sub('᾽','’',l)
        l = re.sub('Αι','Αἰ',l)
        l = re.sub('Ει','Εἰ',l)
        l = re.sub('Οι','Οἰ',l)
        l = re.sub('"Α','Ἄ',l)
        l = re.sub('\\bΡ','Ῥ',l)
        l = re.sub('”\.','῾.',l)
        l = re.sub('\\bΑ([βγδεζηθικλμνξοπρστυχφψω])','Ἀ\g<1>',l)
        l = re.sub('\\bΙ([βγδεζηθικλμνξοπρστυχφψω])','Ἰ\g<1>',l)
        l = re.sub('\\bΕ([βγδεζηθικλμνξοπρστυχφψω])','Ἐ\g<1>',l)
        l = re.sub('\\bΟ([βγδεζηθικλμνξοπρστυχφψω])','Ὀ\g<1>',l)
        l = re.sub('\\bΥ([βγδεζηθικλμνξοπρστυχφψω])','Ὑ\g<1>',l)
        l = re.sub('κτέ','κτἑ',l)
        l = re.sub('Σοφοκλ[ήὴ]ς','Σοφοκλῆς',l)
        #l = re.sub('Α([a-z])','A\g<1>', l)
        l = re.sub("\.'",".’",l)
        l = re.sub('·','·',l)
        #θέμιν’.	replace	θέμιν‘.
        l = re.sub('‘','’',l)
        # ignore mdashes
        #l = re.sub('—',' ', l)
      
        
        l = re.sub("([α-ω])'",'\g<1>’',l)
        l = re.sub("(᾿Ἀ|'Ἀ)","Ἀ",l)
        l = re.sub("\\bν\.",'v.',l)
        l = re.sub('Εust','Eust',l)
        l = re.sub('Ο([a-z])','O\g<1>',l)
        l = re.sub('῾','‘',l)
        l = re.sub('᾿','’',l)
        l = re.sub('[-]([A-ZΑ-Ω])','\g<1>',l)

        l = re.sub('Ἐι','Εἰ',l)
        l = re.sub('Ἀι','Αἰ',l)
        l = re.sub('Ὀι','Οἰ',l)

        l = re.sub('Ἐυ','Εὐ',l)
        l = re.sub('Ἀυ','Αὐ',l)
        l = re.sub('Ὀυ','Οὐ',l)
        l = re.sub('–',' ',l)
        l = re.sub('[ ]+',' ',l)

        if(curbuf):
            curbuf = curbuf + '\n' + l
        else:
            curbuf = l
    curbuf = re.sub('-\s+','',curbuf)

    
    usepage = str(i)
    pagebuf[usepage] = curbuf
    wordindex[usepage] = {}
    for foo in curbuf.split():
        wordindex[usepage][foo] = 1
    f.close()

pageoffset = 32
pageoffset2 = 66
# > 879

f = open('../GRC_misc/nauck.tragfrag.xml')
outf = open('../GRC_misc/nauck.tragfrag-2.xml','w')

for l in f:
    l = re.sub('\s+$','',l)
    m = re.search('([0-9]+\s+)'+g2rreg+'\\b',l)
    if(m):
        #print(m[1],m[2])
        curkey = m[1] + m[2]
        cursub = m[1] + g2r[m[2]]
        l = re.sub(curkey,cursub,l)

    if(re.search('·',l)):
        print('middot',l)
    print(l,file=outf)
outf.close()
f.close()

f = open('../GRC_misc/nauck.tragfrag-2.xml')
nauckbuf = re.sub('᾽','’',nfc(f.read()))
nauckbuf = re.sub('·','·',nauckbuf)
nauckbuf = re.sub('᾽','’',nauckbuf)
nauckbuf = re.sub('—',' ',nauckbuf)
nauckbuflist = nauckbuf.split('<pb n="')
for foo in nauckbuflist:
    m = re.search('(^[0-9IVXLC]+)',foo)
    if(m):
        foo = re.sub('<[/]*add>','',foo)
        foo = re.sub('<[^>]+>',' ',foo)
        foo = re.sub('\s+','\n',foo)
        nauckpages[m[1]] = foo
        xmlindex[m[1]] = {}
        for word in foo.split():
            xmlindex[m[1]][word] = 1
        #print(m[1])
f.close()

middot <l>ἀλλ᾽ ἕν γ' ἔχει τι χρηστόν· ἐν κήδει γὰρ ὢν</l>
middot <l>τρόπον γυναικὸς χρηστὸν ἔνδον λαμβάνειν·</l>


In [5]:
import difflib
tokenlist= []
taglist = {}
def side_by_side_diff(fout,i,a: str, b: str, width=60):
    from itertools import zip_longest
    global tokenlist
    global wordindex
    global xmlindex

    a_lines = a.splitlines()
    b_lines = b.splitlines()
    sm = difflib.SequenceMatcher(None, a_lines, b_lines)

    tokennum = -1
    if(int(i)>947):
        curpage = str(int(i) + 66+3)
    elif(int(i)>879):
        curpage = str(int(i) + 66)
    else:
        curpage = str(int(i) + 32)

    curpage = str(i)

 
        
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        for left, right in zip_longest(a_lines[i1:i2], b_lines[j1:j2], fillvalue=''):
            if(tag == 'replace' and re.sub('[’‘‘”]','',left) == re.sub('[’‘‘”]','',right)):
                tag = 'equal2'
            if( tag == 'delete'):
                if(curpage in wordindex and left in wordindex[curpage]):
                    tag = tag + '2'
            if(tag == 'insert' ):
                if(i in xmlindex and right in xmlindex[i]):
                    tag = tag + '4'

            print(i,left,tag,right,sep='\t',file=fout)
            if(tag in taglist):
                taglist[tag] = taglist[tag] + 1
            else:
                taglist[tag] = 1
            continue
            if(i == '316'):
                print(i,left,tag,right,sep='\t')
            
            #if(left == '‘σὺ'):
            #    print('x',i,left,tag,right,sep='\t')
                
            if( tag == 'delete'):
                if(curpage in wordindex and left in wordindex[curpage]):
                    tag = tag + '2'
            if(tag == 'insert' ):
                if(i in xmlindex and right in xmlindex[i]):
                    tag = tag + '4'
                

            if(left == '' and right in wordindex[curpage] and tag == 'replace'):
                tag = tag + '5'
            if( left in ['—'] and tag == 'delete'):
                tag = 'equal9'
            #if(tag == 'replace' and re.sub('\*','',left) == right):
            #    tag = 'equal*'
            if tag == 'equal':
                marker = ' '
            elif tag == 'replace' or tag == 'delete':
                if(re.sub('[‘῾῾ ̔’,᾿\']','',left) == re.sub("[῾\"'‘”’\']",'',right)):
                    tag = 'equal22'
                if(left.replace('’','') == right):
                    tag = 'equal4'
                if(re.sub('·','',left) == right):
                    tag = 'equal11'
                
                    
                if(curpage in wordindex and left in wordindex[curpage] and not re.search('[0-9]',tag)):
                    #print('a',i,left,tag,right,sep='\t')
                    tag = tag + '8'
                marker = '|'
            elif tag == 'delete':
                marker = '<'
                right = ''
            elif tag == 'insert':
                marker = '>'
                left = ''
            #print(f"{left:<{width}} {marker} {right}")

            if(i == '316'):
                print(i,left,tag,right,sep='\t')

            if(left == '' and right):
                tag = "insert20"

            if(left and right == ''):
                tag = 'insert30'
            print(i,left,tag,right,sep='\t',file=fout)
            tag = re.sub('[0-9]+$','',tag)
            #print(curtoken,left,tag,right,sep='\t',file=fout)
                
fout = open('nauckdiffs.txt','w')
textoff = 32
#if(i>947):
#    textoff = textoff + 3
for i in range(3,959):
    side_by_side_diff(fout,str(i),nauckpages[str(i)],pagebuf[str(i)])
fout.close()

for foo in taglist:
    print(foo,taglist[foo])

replace 18465
equal 251300
equal2 5874
insert4 2047
delete2 1696
delete 496
insert 1039


In [38]:
"/Users/gcrane/github/JupMisc/bsb11538002_text_lines2"


bsbbuf = {}
bsbindex = {}

for i in range(3,989):
    #if(i> 913 and i< 948):
        
    #    continue
    if(i<10):
        fname = '/Users/gcrane/github/JupMisc/bsb11538002_text_lines2/page_00' + str(i) + '.txt'
    elif(i<100):
        fname = '/Users/gcrane/github/JupMisc/bsb11538002_text_lines2/page_0' + str(i) + '.txt'
    else:
        fname = '/Users/gcrane/github/JupMisc/bsb11538002_text_lines2/page_' + str(i) + '.txt'

    
    f = open(fname)
    curbuf = ''
    for l in f:
       # if(re.search('ΑΛΕΑΔΑΙ',l)):
       #     print('0',l)
        while(re.search('([Α-Ω])([A-Z])',l)):
            m = re.search('([Α-Ω])([A-Z])',l)
            if(m):
                #print(m[1],m[2],l)
                curkey = m[1] + m[2]
                curroman = m[1]
                if curroman in r2g:
                    cursub =  m[1] + r2g[curroman]
                else:
                    cursub = m[1] + '<1>' +  m[2]
                l = re.sub(curkey,cursub,l,1)
                #print(fname,'G+R',m[1],m[2],l,'\n')
        l = re.sub('<1>','',l)
        while(re.search('([0-9]+\s+|chol\.\s+)'+g2rreg+'\\b',l)):
            m = re.search('([0-9]+\s+|chol\.\s+)'+g2rreg+'\\b',l)
            if(m):
                #print(fname,'a',m[1],m[2],l)
                curkey = m[1] + m[2]
                cursub = m[1] + g2r[m[2]]
                l = re.sub(curkey,cursub,l,1)
       # if(re.search('ΑΛΕΑΔAI',l)):
       #     print('1a',l)
       # elif(re.search('ΑΛΕΑΔΑΙ',l)):
       #     print('stillok')
        while(re.search('\\b'+g2rreg+'(\\b|[:,\.a-zA-Z])',l)):
            m = re.search('\\b'+g2rreg+'(\\b|[:,\.a-zA-Z])',l)
            if(m):
                #print(m[1],m[2])
                curkey = m[1] + m[2]
                cursub = g2r[m[1]] + m[2] 
                l = re.sub(curkey,cursub,l,1)
       # if(re.search('ΑΛΕΑΔAI',l)):
       #     print('1',l)
        l = re.sub('\s+$','',l)
        if(re.search('(\\b[ΧΙ]+\\b)',l)):
            #print('rn',fname,l)
            while(re.search(g2rreg,l)):
                m = re.search(g2rreg,l)
                subval = g2r[m[1]]
                l = re.sub(m[1],subval,l)
            
        l = re.sub('\\bef\\b','cf',l)
        l = re.sub('[MΜ][AΑ]','MA',l)
        l = re.sub('˙','·',l)
        l = re.sub('·','·',l)
        l = re.sub('—','',l)
        l = re.sub('”,','’,',l)
        l = re.sub('^\s+','',l)
        l = re.sub('\s+','\n',l)
        l = re.sub('᾽','’',l)
        l = re.sub('Αι','Αἰ',l)
        l = re.sub('Ει','Εἰ',l)
        l = re.sub('Οι','Οἰ',l)
        l = re.sub('"Α','Ἄ',l)
        l = re.sub('\\bΡ','Ῥ',l)
        l = re.sub('”\.','῾.',l)
        l = re.sub('\\bΑ([βγδεζηθικλμνξοπρστυχφψω])','Ἀ\g<1>',l)
        l = re.sub('\\bΙ([βγδεζηθικλμνξοπρστυχφψω])','Ἰ\g<1>',l)
        l = re.sub('\\bΕ([βγδεζηθικλμνξοπρστυχφψω])','Ἐ\g<1>',l)
        l = re.sub('\\bΟ([βγδεζηθικλμνξοπρστυχφψω])','Ὀ\g<1>',l)
        l = re.sub('\\bΥ([βγδεζηθικλμνξοπρστυχφψω])','Ὑ\g<1>',l)
        l = re.sub('κτέ','κτἑ',l)
        l = re.sub('Σοφοκλ[ήὴ]ς','Σοφοκλῆς',l)
        #l = re.sub('Α([a-z])','A\g<1>', l)
        l = re.sub("\.'",".’",l)
        l = re.sub('·','·',l)
        #θέμιν’.	replace	θέμιν‘.
        l = re.sub('‘','’',l)
        # ignore mdashes
        #l = re.sub('—',' ', l)
      
        
        l = re.sub("([α-ω])'",'\g<1>’',l)
        l = re.sub("(᾿Ἀ|'Ἀ)","Ἀ",l)
        l = re.sub("\\bν\.",'v.',l)
        l = re.sub('Εust','Eust',l)
        l = re.sub('Ο([a-z])','O\g<1>',l)
        l = re.sub('῾','‘',l)
        l = re.sub('᾿','’',l)
        l = re.sub('[-]([A-ZΑ-Ω])','\g<1>',l)

        l = re.sub('Ἐι','Εἰ',l)
        l = re.sub('Ἀι','Αἰ',l)
        l = re.sub('Ὀι','Οἰ',l)

        l = re.sub('Ἐυ','Εὐ',l)
        l = re.sub('Ἀυ','Αὐ',l)
        l = re.sub('Ὀυ','Οὐ',l)
        l = re.sub('–',' ',l)
        l = re.sub('[ ]+',' ',l)
        l = re.sub('\s+([\)])','\g<1>',l)
        l = re.sub('([\(])\s+','\g<1>',l)
        l = re.sub('([a-zA-Z0-9])([,\.;:])','\g<1>\n\g<2>',l)

        if(curbuf):
            curbuf = curbuf + '\n' + l
        else:
            curbuf = l
    curbuf = re.sub('-\s+','',curbuf)

    
    usepage = str(i)
    bsbbuf[usepage] = curbuf
    bsbindex[usepage] = {}
    for foo in curbuf.split():
        bsbindex[usepage][foo] = 1
    f.close()


In [39]:
import difflib
tokenlist= []
taglist = {}
def side_by_side_diff(fout,i,a: str, b: str, width=60):
    from itertools import zip_longest
    global tokenlist
    global wordindex
    global xmlindex

    a_lines = a.splitlines()
    b_lines = b.splitlines()
    sm = difflib.SequenceMatcher(None, a_lines, b_lines)

    tokennum = -1

    curpage = str(i)

 
        
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        for left, right in zip_longest(a_lines[i1:i2], b_lines[j1:j2], fillvalue=''):
            if(tag == 'replace' and re.sub('[’‘‘”]','',left) == re.sub('[’‘‘”]','',right)):
                tag = 'equal2'
            if( tag == 'delete'):
                if(curpage in wordindex and left in wordindex[curpage]):
                    tag = tag + '2'
            if(tag == 'insert' ):
                if(i in xmlindex and right in xmlindex[i]):
                    tag = tag + '4'

            print(i,left,tag,right,sep='\t',file=fout)
            if(tag in taglist):
                taglist[tag] = taglist[tag] + 1
            else:
                taglist[tag] = 1
            continue

fout = open('nauckdiffs-bsb.txt','w')
textoff = 30
#if(i>947):
#    textoff = textoff + 3
for i in range(3,959):
    curnauck = re.sub('([\,;\.])',' \g<1>',nauckpages[str(i)])
    curnauck = nauckpages[str(i)]
    side_by_side_diff(fout,str(i),curnauck,bsbbuf[str(i+30)])
fout.close()

for foo in taglist:
    print(foo,taglist[foo])

replace 18480
equal 344162
equal2 6530
insert4 3545
delete2 1950
delete 1191
insert 1574


In [41]:
nfd('῞Αἰδου')

'῞Αἰδου'